### Importing Necessary Libraries

In [2]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import pos_tag
from nltk.chunk import ne_chunk

In [3]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\cfl602\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\cfl602\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\cfl602\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\cfl602\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


True

### Step 1: Preprocessing – tokenise and apply PoS tagging

In [ ]:
def preprocess_ner(text):
    sentences = sent_tokenize(text)
    pos_tagged_sentences = []
    for sent in sentences:
        tokens = word_tokenize(sent)
        pos_tagged = pos_tag(tokens)
        pos_tagged_sentences.append(pos_tagged)
    return pos_tagged_sentences

### Step 2: Feature extraction – use nltk.ne_chunk to identify entities

In [8]:
def extract_entities(pos_tagged_sentences):
    entities = []
    for sent in pos_tagged_sentences:
        chunked = ne_chunk(sent)   # returns a nested nltk.Tree object
        for subtree in chunked:
            if hasattr(subtree, 'label') and subtree.label() in ['PERSON', 'GPE', 'ORGANIZATION']:
                entity_name = ' '.join([leaf[0] for leaf in subtree.leaves()])
                entities.append((entity_name, subtree.label()))
    return entities

### Step 3: Data engineering – filter and group into specific categories

In [9]:
def group_entities(entities):
    """Group entities by type (PERSON, GPE, ORGANISATION)."""
    grouped = {
        'PERSON': [],
        'GPE': [],        # Geopolitical Entity (city, country, region)
        'ORGANIZATION': []
    }
    for name, label in entities:
        if label in grouped:
            grouped[label].append(name)
    return grouped

### Step 4: Pattern discovery – analyse most frequent entities

In [10]:
def analyse_frequency(grouped_entities):
    """Count and display the most common entities in each category."""
    print("\n" + "="*50)
    print("PATTERN DISCOVERY – Most Frequent Entities")
    print("="*50)
    for category, entities_list in grouped_entities.items():
        if entities_list:
            freq = nltk.FreqDist(entities_list)
            top3 = freq.most_common(3)
            print(f"\n{category} – top 3:")
            for entity, count in top3:
                print(f"  • {entity}: {count} times")
        else:
            print(f"\n{category}: No entities found.")

In [12]:
sample_news = """
Apple Inc. CEO Tim Cook announced a new AI research centre in Seattle. 
Microsoft President Brad Smith also attended the event. 
The initiative aims to compete with Google's DeepMind in London.
"""

print("Input news text:")
print(sample_news)

Input news text:

Apple Inc. CEO Tim Cook announced a new AI research centre in Seattle. 
Microsoft President Brad Smith also attended the event. 
The initiative aims to compete with Google's DeepMind in London.



In [13]:
# Step 1: Preprocessing (tokenise + PoS tagging)
pos_tagged_sents = preprocess_ner(sample_news)

In [14]:
# Step 2: Extract named entities
all_entities = extract_entities(pos_tagged_sents)
print("\nExtracted Named Entities:", all_entities)


Extracted Named Entities: [('Apple', 'PERSON'), ('Inc.', 'ORGANIZATION'), ('Tim Cook', 'PERSON'), ('Seattle', 'GPE'), ('Microsoft', 'PERSON'), ('Brad Smith', 'PERSON'), ('Google', 'PERSON'), ('DeepMind', 'ORGANIZATION'), ('London', 'GPE')]


In [15]:
# Step 3: Group into categories
grouped = group_entities(all_entities)
print("\nGrouped entities:")
for cat, ents in grouped.items():
    print(f"  {cat}: {ents}")


Grouped entities:
  PERSON: ['Apple', 'Tim Cook', 'Microsoft', 'Brad Smith', 'Google']
  GPE: ['Seattle', 'London']
  ORGANIZATION: ['Inc.', 'DeepMind']


In [16]:
# Step 4: Pattern discovery – most frequent subjects
analyse_frequency(grouped)


PATTERN DISCOVERY – Most Frequent Entities

PERSON – top 3:
  • Apple: 1 times
  • Tim Cook: 1 times
  • Microsoft: 1 times

GPE – top 3:
  • Seattle: 1 times
  • London: 1 times

ORGANIZATION – top 3:
  • Inc.: 1 times
  • DeepMind: 1 times
